# Week 31, Day 5: __slots__ and memory

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w031d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Point:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y

p = Point(1, 2)
p.x = 10
try:
    p.z = 3
except AttributeError:
    print("no z")
print(p.x, hasattr(p, "__dict__"))

In [ ]:
class Reading:
    def __init__(self, sensor, celsius):
        self.sensor = sensor
        self.celsius = celsius

r = Reading("kitchen", 21.5)
print(vars(r))
r.note = "door open"            # any object can grow a new attribute
r.celsuis = 30                  # including a misspelt one, silently
print(vars(r))
print("__init__" in vars(Reading), "sensor" in vars(Reading))

In [ ]:
import tracemalloc

class Plain:
    def __init__(self, x, y, z):
        self.x, self.y, self.z = x, y, z

class Slotted:
    __slots__ = ("x", "y", "z")

    def __init__(self, x, y, z):
        self.x, self.y, self.z = x, y, z

def bytes_each(cls, n=50_000):
    tracemalloc.start()
    objects = [cls(i, i, i) for i in range(n)]
    used, _ = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return used // n

print("plain:  ", bytes_each(Plain), "bytes per object")
print("slotted:", bytes_each(Slotted), "bytes per object")

In [ ]:
class Reading:
    __slots__ = ("sensor", "celsius")

    def __init__(self, sensor, celsius):
        self.sensor = sensor
        self.celsius = celsius

    def __repr__(self):
        return f"Reading({self.sensor!r}, {self.celsius})"

r = Reading("kitchen", 21.5)
r.celsius = 22.0                     # changing a slot is fine
print(r)
for name, value in [("celsuis", 30), ("note", "door open")]:
    try:
        setattr(r, name, value)
    except AttributeError as err:
        print("AttributeError:", err)
try:
    vars(r)
except TypeError as err:
    print("TypeError:", err)

In [ ]:
class Reading:
    __slots__ = ("sensor", "celsius")

    def __init__(self, sensor, celsius):
        self.sensor, self.celsius = sensor, celsius

class Loose(Reading):
    pass

class Humid(Reading):
    __slots__ = ("humidity",)          # only the NEW name

    def __init__(self, sensor, celsius, humidity):
        super().__init__(sensor, celsius)
        self.humidity = humidity

a = Loose("hall", 19.0)
a.anything = 1                         # Loose has a __dict__ again
h = Humid("bathroom", 23.5, 80)
print(vars(a), hasattr(h, "__dict__"), h.sensor, h.humidity)

In [ ]:
def slot_names(obj):
    names = []
    for cls in reversed(type(obj).__mro__):
        names.extend(getattr(cls, "__slots__", ()))
    return names

class Reading:
    __slots__ = ("sensor", "celsius")
    def __init__(self, sensor, celsius):
        self.sensor, self.celsius = sensor, celsius

class Humid(Reading):
    __slots__ = ("humidity",)
    def __init__(self, sensor, celsius, humidity):
        super().__init__(sensor, celsius)
        self.humidity = humidity

h = Humid("bathroom", 23.5, 80)
print({name: getattr(h, name) for name in slot_names(h)})

In [ ]:
class Coord:
    __slots__ = ("lat", "lon")

    def __init__(self, lat, lon):
        object.__setattr__(self, "lat", lat)       # bypass our own __setattr__
        object.__setattr__(self, "lon", lon)

    def __setattr__(self, name, value):
        raise AttributeError(f"Coord is immutable: can't set {name}")

    def __eq__(self, other):
        if not isinstance(other, Coord):
            return NotImplemented
        return (self.lat, self.lon) == (other.lat, other.lon)

    def __hash__(self):
        return hash((self.lat, self.lon))

    def __repr__(self):
        return f"Coord({self.lat}, {self.lon})"

lagos = Coord(6.45, 3.39)
print(lagos, {lagos: "Lagos"}[Coord(6.45, 3.39)])
try:
    lagos.lat = 0
except AttributeError as err:
    print("AttributeError:", err)

> **Key idea.** Ordinary objects keep their attributes in a per-object `__dict__` (shown by `vars`), which allows new attributes at any time but costs memory and lets typos pass. `sys.getsizeof` measures one object shallowly; `tracemalloc` measures what a batch of objects really uses. `__slots__ = ("a", "b")` gives a fixed set of attributes and no `__dict__`: less memory and typos caught, but no new attributes, no `vars()`, and no class attribute with a slot's name. Every class in a family needs `__slots__` (only its new names; `()` for mixins), and two parents with non-empty slots can't be combined. Overriding `__setattr__` to raise, and setting values with `object.__setattr__` in `__init__`, makes objects immutable.

### Exercise: A slotted pixel

Write `Pixel(x, y, colour)` with `__slots__` for its three attributes, and a `repr` like `Pixel(3, 4, '#ff0000')`. Objects must have no `__dict__`, and assigning any other attribute must raise `AttributeError`. `colour` defaults to `"#000000"`; don't make it a class attribute.

In [ ]:
class Pixel:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pixel():\n    \"pixels\"\n    P = ns[\"Pixel\"]\n    p = P(3, 4, \"#ff0000\")\n    assert (p.x, p.y, p.colour) == (3, 4, \"#ff0000\") and repr(p) == \"Pixel(3, 4, '#ff0000')\"\n    assert P(0, 0).colour == \"#000000\" and not hasattr(p, \"__dict__\")\n    p.colour = \"#00ff00\"\n    assert p.colour == \"#00ff00\"\n    try:\n        p.color = \"#0000ff\"\n        assert False, \"a misspelt attribute should raise AttributeError\"\n    except AttributeError:\n        pass\n    assert set(P.__slots__) == {\"x\", \"y\", \"colour\"}")

### Exercise: A slotted family

Write a family of shapes where **no** object has a `__dict__`:

- `Shape(name)` with a slot for `name`, and `describe()` returning `"Circle 'wheel': area 3.14"` (class name, the name with `repr`, and `area()` to 2 places);
- `Circle(name, radius)`, `Rect(name, width, height)` and `Square(name, side)` (a kind of `Rect` with width and height both `side`, adding no new attributes).

Each class lists only its own new slots, uses `super().__init__`, and inherits `describe`.

In [ ]:
import math

class Shape:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_shapes():\n    \"shapes\"\n    C, R, S = ns[\"Circle\"], ns[\"Rect\"], ns[\"Square\"]\n    assert C(\"wheel\", 1).describe() == \"Circle 'wheel': area 3.14\" and R(\"door\", 2, 0.5).describe() == \"Rect 'door': area 1.00\"\n    assert S(\"tile\", 3).describe() == \"Square 'tile': area 9.00\" and (S(\"t\", 3).width, S(\"t\", 3).height) == (3, 3)\ndef test_slots():\n    \"slots in every class\"\n    Sh, C, R, S = ns[\"Shape\"], ns[\"Circle\"], ns[\"Rect\"], ns[\"Square\"]\n    for obj in [Sh(\"x\"), C(\"x\", 1), R(\"x\", 1, 2), S(\"x\", 1)]:\n        assert not hasattr(obj, \"__dict__\"), f\"{type(obj).__name__} objects shouldn't have a __dict__\"\n    assert tuple(Sh.__slots__) == (\"name\",) and tuple(C.__slots__) == (\"radius\",)\n    assert set(R.__slots__) == {\"width\", \"height\"} and tuple(S.__slots__) == (), \"list only new names in each class\"\n    assert issubclass(S, R) and all(\"describe\" not in vars(k) for k in (C, R, S))")

### Exercise: A mixin for slotted classes

Write `SlotsMixin`, a mixin with **empty** slots that gives any slotted class three things, using the slot names from every class in its MRO (parents' names first):

- `__repr__`: `Point(x=1, y=2)`, with each value shown by `repr`;
- `__eq__`: equal when the other object has exactly the same class and all slot values are equal (otherwise `NotImplemented`);
- `as_dict()`: a dictionary from slot name to value.

Then write `Point(x, y)` and `Point3(x, y, z)` (a kind of `Point`) using it, with no `__dict__`.

In [ ]:
class SlotsMixin:
    ___

class Point(SlotsMixin):
    ___

class Point3(Point):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_mixin():\n    \"the mixin\"\n    M, P, P3 = ns[\"SlotsMixin\"], ns[\"Point\"], ns[\"Point3\"]\n    assert tuple(M.__slots__) == () and not hasattr(P(1, 2), \"__dict__\") and not hasattr(P3(1, 2, 3), \"__dict__\")\n    assert repr(P(1, 2)) == \"Point(x=1, y=2)\" and repr(P3(1, \"a\", 3)) == \"Point3(x=1, y='a', z=3)\"\n    assert P3(1, 2, 3).as_dict() == {\"x\": 1, \"y\": 2, \"z\": 3}\n    assert P(1, 2) == P(1, 2) and P(1, 2) != P(2, 1) and P(1, 2) != P3(1, 2, 0) and P(1, 2).__eq__((1, 2)) is NotImplemented\n    class Pair(M):\n        __slots__ = (\"a\", \"b\")\n        def __init__(self, a, b):\n            self.a, self.b = a, b\n    assert repr(Pair(5, [1])) == \"Pair(a=5, b=[1])\"")

### Exercise: A frozen colour

Write `Colour(r, g, b)` (each a whole number from 0 to 255, otherwise `ValueError`), slotted and **immutable**: any assignment after it's made raises `AttributeError`, and so does deleting an attribute (`del c.r`; that calls `__delattr__`). It has `==`, a matching `__hash__`, `repr` like `Colour(255, 128, 0)`, and `mix(other)`, returning a **new** colour with each channel the average of the two, rounded down.

In [ ]:
class Colour:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_colour():\n    \"colours\"\n    C = ns[\"Colour\"]\n    c = C(255, 128, 0)\n    assert repr(c) == \"Colour(255, 128, 0)\" and c == C(255, 128, 0) and c != C(255, 128, 1) and len({c, C(255, 128, 0)}) == 1\n    assert c.mix(C(0, 0, 255)) == C(127, 64, 127) and not hasattr(c, \"__dict__\")\n    for f in [lambda: setattr(c, \"r\", 0), lambda: setattr(c, \"alpha\", 1), lambda: delattr(c, \"g\")]:\n        try:\n            f()\n            assert False, \"colours can't be changed\"\n        except AttributeError:\n            pass\n    assert c == C(255, 128, 0)\n    for bad in [(256, 0, 0), (0, -1, 0), (0, 0, 1.5), (True, 0, 0)]:\n        try:\n            C(*bad)\n            assert False, f\"Colour{bad} should raise ValueError\"\n        except ValueError:\n            pass")

### Bug hunt: A default that clashes

The program crashes before it starts, with `ValueError: 'size' in __slots__ conflicts with class variable`. Keep the slots and the default size of 10, and fix it so it prints `10 14`.

In [ ]:
class Font:
    __slots__ = ("name", "size")
    size = 10

    def __init__(self, name):
        self.name = name

body = Font("Georgia")
title = Font("Georgia")
title.size = 14
print(body.size, title.size)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"sizes\"\n    expect_output(\"10 14\")\n    assert not hasattr(ns[\"Font\"](\"x\"), \"__dict__\")")

### Bug hunt: The typo that got through

`Parcel` uses slots so that misspelt attributes are caught, but `Express` parcels let a typo through: this prints `weight not changed: 500` instead of `typo caught`. Fix `Express`.

In [ ]:
class Parcel:
    __slots__ = ("code", "weight")

    def __init__(self, code, weight):
        self.code = code
        self.weight = weight

class Express(Parcel):
    def label(self):
        return f"EXPRESS {self.code}"

e = Express("EX1", 500)
try:
    e.wieght = 750
    print("weight not changed:", e.weight)
except AttributeError:
    print("typo caught")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"caught\"\n    expect_output(\"typo caught\")\n    assert ns[\"Express\"](\"A\", 1).label() == \"EXPRESS A\"")

### Bug hunt: vars on a slotted object

After `Sample` was given slots, `show` crashes with `TypeError: vars() argument must have __dict__ attribute`. Fix `show` (not the class) so it prints `site=Kano, ppm=412`.

In [ ]:
class Sample:
    __slots__ = ("site", "ppm")

    def __init__(self, site, ppm):
        self.site = site
        self.ppm = ppm

def show(obj):
    return ", ".join(f"{k}={v}" for k, v in vars(obj).items())

print(show(Sample("Kano", 412)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"shown\"\n    expect_output(\"site=Kano, ppm=412\")\n    assert \"__slots__\" in vars(ns[\"Sample\"])")

### Puzzle: A slotted reading

Arrange the lines into a class whose objects hold exactly a station name and a rainfall amount. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
class A:
    __slots__ = ("x",)
    def __init__(self):
        self.x = 1

class B(A):
    pass

class C(A):
    __slots__ = ("y",)
    def __init__(self):
        super().__init__()
        self.y = 2

p = ___   # hasattr(A(), "__dict__")
q = ___   # hasattr(B(), "__dict__")
r = ___   # hasattr(C(), "__dict__")
s = ___   # C.__slots__
t = ___   # [k for cls in reversed(C.__mro__) for k in getattr(cls, "__slots__", ())]

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (False, True, False), \"p to r\"\n    assert tuple(s) == (\"y\",) and t == [\"x\", \"y\"]")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a vector that behaves

Write `Vec(x, y)`, a 2D vector that uses most of this week's special methods. It must be slotted (no `__dict__`) and immutable (assigning or deleting any attribute raises `AttributeError`).

- `repr` gives `Vec(3, 4)`. `==` and `hash` use both coordinates.
- `v + w` and `v - w` with another `Vec`; `v * k` and `k * v` with a number. Anything else gives `TypeError` (return `NotImplemented`).
- `abs(v)` is the length (`math.hypot(x, y)`); `bool(v)` is False only for `Vec(0, 0)`.
- It's a small sequence: `len(v)` is 2, `v[0]` and `v[1]` (and `v[-1]`, `v[-2]`) give the coordinates, any other index raises `IndexError`, and iterating gives `x` then `y`, so `x, y = v` and `list(v)` work.
- Vectors have **no** order: `Vec(1, 2) < Vec(3, 4)` raises `TypeError`.

Then write `centre(vecs)`, the mean of a list of vectors as a `Vec`, raising `ValueError` for an empty list.

In [ ]:
import math

class Vec:
    ___

def centre(vecs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def V(*a):\n    return ns[\"Vec\"](*a)\ndef test_basics():\n    \"repr, equality, immutability\"\n    v = V(3, 4)\n    assert repr(v) == \"Vec(3, 4)\" and v == V(3, 4) and v != V(4, 3) and len({v, V(3, 4)}) == 1\n    assert not hasattr(v, \"__dict__\")\n    for f in [lambda: setattr(v, \"x\", 0), lambda: setattr(v, \"z\", 1), lambda: delattr(v, \"y\")]:\n        try:\n            f()\n            assert False, \"vectors can't be changed\"\n        except AttributeError:\n            pass\n    assert v == V(3, 4)\ndef test_maths():\n    \"arithmetic\"\n    v, w = V(3, 4), V(1, -2)\n    assert v + w == V(4, 2) and v - w == V(2, 6) and v * 2 == V(6, 8) and 0.5 * v == V(1.5, 2.0)\n    assert abs(v) == 5.0 and bool(v) and not V(0, 0)\n    for f in [lambda: v + 1, lambda: v * w, lambda: \"a\" * v, lambda: v < w]:\n        try:\n            f()\n            assert False, \"should raise TypeError\"\n        except TypeError:\n            pass\ndef test_sequence():\n    \"a small sequence\"\n    v = V(3, 4)\n    x, y = v\n    assert (x, y) == (3, 4) and list(v) == [3, 4] and len(v) == 2 and v[0] == 3 and v[-1] == 4 and v[-2] == 3\n    for i in [2, -3]:\n        try:\n            v[i]\n            assert False, f\"v[{i}] should raise IndexError\"\n        except IndexError:\n            pass\ndef test_centre():\n    \"the centre\"\n    assert centre([V(0, 0), V(4, 0), V(4, 2), V(0, 2)]) == V(2, 1)\n    assert centre([V(1, 1)]) == V(1, 1)\n    try:\n        centre([])\n        assert False, \"an empty list should raise ValueError\"\n    except ValueError:\n        pass")

## Warm-up

`Point` lists its attributes in `__slots__`, so its objects can hold `x` and `y` and nothing else. Changing `x` is fine; adding `z` isn't. And the object has no `__dict__`, the dictionary that every other object you've made carries around. Today is about that dictionary, what it costs, and when it's worth giving up.

## Where attributes live

An ordinary object stores its attributes in a dictionary of its own, `obj.__dict__`, which `vars(obj)` shows (you've used it since Week 28). Methods and class attributes live in the **class's** dictionary, shared by every object, so each object only pays for its own attribute values.

That flexibility has two costs:

- **Memory.** A dictionary per object is more than the values alone need. One object doesn't matter; half a million do.
- **Typos pass silently.** `r.celsuis = 30` created a new attribute instead of failing. The real `celsius` is unchanged, and the bug shows up somewhere else, later.

## Measuring memory

`sys.getsizeof(x)` gives the size in bytes of one object **on its own**, not counting the objects it refers to. That makes it easy to misread: it doesn't count a list's items, a dictionary's keys and values, or (in recent Pythons) an object's attribute storage properly. To see what a program really uses, measure a large batch of objects with `tracemalloc`, the standard library's memory tracer:

The numbers include the list holding the objects and the integers inside them, so they're "the cost of one more object" in a realistic program. The exact figures depend on the Python version, but slotted objects come out clearly smaller. Recent versions of Python have made ordinary objects much leaner than they used to be, so the saving is smaller than older books and articles describe. Measure before you decide it matters.

## __slots__

A class attribute named `__slots__`, holding a tuple of attribute names, tells Python to give each object a fixed place for exactly those attributes, and **no** `__dict__`.

What you get:

- **Less memory per object**, which matters when there are very many.
- **Typos caught**: assigning an attribute that isn't a slot raises `AttributeError` at once.
- Slightly **faster** attribute access (a small effect; don't use slots for speed alone).

What you give up:

- **No new attributes** on objects, ever, even ones that would be useful later.
- **No `__dict__`**, so `vars(obj)` fails, and code (yours or a library's) that relies on it breaks. Read slot values with `getattr(obj, name)` instead.
- **No class attribute with a slot's name.** A default like `size = 10` in the class body clashes with a slot called `size`, and Python refuses to create the class. Set defaults in `__init__` instead.

So slots are for classes with a fixed, known set of attributes that you'll make **in large numbers**: points, pixels, readings, cells of a grid, records loaded from a big file. For a class with a handful of objects, the dictionary's flexibility is worth more. Week 32's dataclasses can add slots for you with `slots=True`.

## Slots and inheritance

Slots are declared per class, and a child class gets a `__dict__` back unless it declares `__slots__` too:

The rules:

- Every class in the family must declare `__slots__` for objects to have no `__dict__`. A child with nothing new to add writes `__slots__ = ()`.
- A child lists only its **new** attributes. Repeating a parent's slot wastes space and hides the parent's.
- **Mixins** that hold no data should declare `__slots__ = ()`, so they don't bring a `__dict__` with them.
- Two parents that **both** have non-empty slots can't be combined: `class C(A, B)` fails with "multiple bases have instance lay-out conflict". Keep slots in one branch of a family, and empty slots in mixins.

Code that wants to list a slotted object's attributes can walk the MRO (Week 30, Day 3), because each class's `__slots__` holds only its own names:

## Values that can't change

Slots fix **which** attributes an object has, not their **values**. For value objects that must never change, such as anything used as a dictionary key (Week 29, Day 2), you can go one step further. `obj.name = value` calls `type(obj).__setattr__(obj, "name", value)`, so a class can refuse every assignment, and set its own values in `__init__` by calling `object`'s version directly:

Compared with Week 29's read-only properties, this needs no property per attribute, and with slots there's no `__dict__` to change behind its back either. (A determined programmer can still call `object.__setattr__` themselves: Python trusts adults. The aim is to stop accidents, not attacks.) Week 32's `@dataclass(frozen=True)` writes this for you.

## Practice

## Quiz

## Daily challenge

## Before you go

Think of a program you might write that makes very many small objects of one kind (readings from a sensor, rows of a large file, cells of a game board), and decide whether slots would be worth it there. Tomorrow is the Forge: a week of special methods, put together.